# Building Agents with custom Model Context Protocol (MCP) Tools

## Overview

This notebook demonstrates how to build and integrate a custom **Model Context Protocol (MCP)** server with Google **Agent Development Kit (ADK)** to build AI agents.

The **Model Context Protocol (MCP)** is an open standard that enables AI models and agents to interact with external data sources, APIs, and tools through a uniform protocol. Instead of writing custom tool integration logic for every new data source, MCP allows developers to expose tools via standard servers (using transport mechanisms like Server-Sent Events / SSE or Standard I/O) that any compatible agent can connect to.

In this tutorial, we will:
* Build an **MCP Server** (`mcp_server.py`) using the `FastMCP` SDK that exposes tools to query and update inventory.
* Configure an **ADK Agent** (`LlmAgent`) that connects to the MCP Server over SSE using `McpToolset` and `SseServerParams`.
* Interact with the MCP-powered agent using the **ADK Developer UI** (`adk web`) to inspect tool calls, test conversational workflows, and verify inventory modifications.
* Additionally, we'll deploy the Inventory MCP Server to **Google Cloud Run** as a production microservice and connect a remote ADK agent to it.

## Learning Objectives

By the end of this notebook, you will understand how to:
* **Understand MCP Architecture:** Learn how MCP separates tool/data service providers (servers) from AI agent consumers (clients).
* **Build an MCP Server:** Use `FastMCP` (`@mcp.tool()`) to expose Python functions and schemas as standardized MCP tools.
* **Integrate MCP Tools in ADK:** Connect an ADK `LlmAgent` to an MCP server using `McpToolset` and `SseServerParams`.
* **Use ADK Developer UI:** Run `adk web` to inspect and interactively test MCP-integrated agents.
* **Deploy MCP Server to Cloud Run:** Package and deploy your MCP server to Google Cloud Run and connect remote ADK agents to its `/sse` HTTPS endpoint.


## Setup and Imports

Let's start by importing the necessary Python libraries for ADK, MCP, and asynchronous execution.


In [ ]:
import asyncio
import csv
import importlib
import logging
import os
import socket
import subprocess
import sys
import time
import warnings

import pandas as pd
import google.auth
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.genai import types

if "." not in sys.path:
    sys.path.insert(0, ".")

# Ignore warnings and opentelemetry logs for a clean notebook output
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry.trace").setLevel(logging.CRITICAL)


In [ ]:
_, PROJECT_ID = google.auth.default()
GEMINI_LOCATION = "global"
CLOUD_RUN_LOCATION = "us-central1"
MODEL = "gemini-3.5-flash"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["CLOUD_RUN_LOCATION"] = CLOUD_RUN_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"


## Build Inventory MCP Agent

In this section, we will build the complete local MCP tool agent pipeline: creating the inventory dataset, building the MCP server with read and mutation tools, starting the local SSE server, defining the ADK agent, and testing it interactively with the Developer UI (`adk web`).


In [ ]:
!mkdir -p ./custom_mcp_agent/agent


In [ ]:
%%writefile ./custom_mcp_agent/agent/__init__.py
# pylint: skip-file
from . import agent


### Create Agent Environment Configuration (`.env`)

Since our agent module `./custom_mcp_agent` is located outside of the default `adk_agents` directory, we create a `.env` file inside `./custom_mcp_agent` so that ADK automatically loads our Vertex AI environment configuration when running `adk web`.


In [ ]:
%%writefile ./custom_mcp_agent/.env
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE


### Create Sample Inventory Data (`sku_data.csv`)

In this example, our MCP server will manage an inventory database stored in a CSV file (`sku_data.csv`).
Each record contains details such as the product SKU, name, category, quantity on hand, unit price, and stock status.

We'll create a module directory `./custom_mcp_agent` and write our sample CSV file there.


In [ ]:
%%writefile ./custom_mcp_agent/sku_data.csv
SKU,ProductName,Category,SupplierID,SupplierName,QuantityOnHand,ReorderLevel,UnitPrice,UnitCost,LastStockInDate,LastStockOutDate,Location,Status,Notes
SKU001,"Laptop Pro 15""",Electronics,SUP001,TechGlobal,60,5,1200.00,950.00,2023-04-10,2023-05-15,Warehouse A - Shelf 1,Low Stock,High-performance model
SKU002,Wireless Mouse,Accessories,SUP002,OfficeGear,70,20,25.00,15.00,2023-03-20,2023-05-10,Warehouse B - Bin 3,Low Stock,Ergonomic design
SKU003,Mechanical Keyboard,Accessories,SUP001,TechGlobal,40,10,75.00,50.00,2023-04-01,2023-05-05,Warehouse A - Shelf 2,In Stock,RGB Backlit
SKU004,"27"" 4K Monitor",Electronics,SUP003,DisplayInc,25,5,300.00,220.00,2023-02-15,2023-04-28,Warehouse C - Area 1,In Stock,
SKU005,HD Webcam,Accessories,SUP002,OfficeGear,60,15,50.00,30.00,2023-03-05,2023-05-12,Warehouse B - Bin 5,In Stock,Includes microphone
SKU006,USB-C Hub,Accessories,SUP001,TechGlobal,30,10,35.00,20.00,2023-04-18,2023-05-01,Warehouse A - Shelf 3,Low Stock,7-in-1 adapter


### Build the Inventory MCP Server (`mcp_server.py`)

Now we build the MCP Server using the `FastMCP` SDK from the `mcp.server.fastmcp` package.
We will break down the implementation into three sections:
1. **FastMCP Server Initialization**: Creating the server instance and specifying host/port.
2. **Inventory Query Tool (`list_skus`)**: Exposing a read-only tool to inspect SKUs from the CSV file.
3. **Inventory Mutation Tool (`update_sku_qty`) & Entrypoint**: Exposing a state-mutating tool to update product quantities and launching the Server-Sent Events (SSE) server.


#### FastMCP Server Initialization

We start by importing the required libraries (`asyncio`, `csv`, `os`, `FastMCP`, `Field`) and initializing a `FastMCP` server instance from the `mcp.server.fastmcp` SDK.

* **`FastMCP(...)`**: Initializes the server with a descriptive `name` and detailed natural-language `instructions` that help AI clients understand the overall purpose of this MCP server.
* **Network Configuration**: We configure the server to listen on host `0.0.0.0` and port `4200` by default (or the port specified by the `PORT` environment variable when deployed to Cloud Run).

In [ ]:
%%writefile ./custom_mcp_agent/mcp_server.py
"""MCP Server for inventory management."""

import csv
import os

from mcp.server.fastmcp import FastMCP
from pydantic import Field

# Define the path to the CSV file relative to this script
CSV_FILE_PATH = os.path.join(os.path.dirname(__file__), "sku_data.csv")

mcp = FastMCP(
    name="Inventory MCP Server",
    instructions="""
    This Server provides inventory related data and helps in
    updating the quantity of a specific SKU.

    Call list_skus() to get the list of all the SKUs or any details related to
    the items/SKUs.
    Call update_sku_qty(sku_id, quantity, sign) to update the quantity of a specific SKU.
    """,
    host="0.0.0.0",
    port=int(os.environ.get("PORT", 4200)),
)


#### Inventory Query Tool (`list_skus`)

Next, we define our first MCP tool, `list_skus`, which allows an agent to query inventory items.

* **`@mcp.tool()` Decorator**: Registers the Python function as an MCP tool on our FastMCP server.
* **Pydantic Schema Annotation**: We use Pydantic's `Field("*", description="Name of the SKU")` for parameter description. The MCP server automatically translates this into a JSON Schema so connecting agents know what arguments the tool accepts.
* **Docstring Guidance**: Notice the detailed docstring describing *when* to call the function and *what* it returns. The LLM reads this docstring via MCP to decide when to invoke the tool.
* **CSV Reading Logic**: Opens `sku_data.csv` using `csv.DictReader` and returns all records as a list of dictionaries.

In [ ]:
%%writefile -a ./custom_mcp_agent/mcp_server.py


@mcp.tool()
def list_skus(sku_name: str = Field("*", description="Name of the SKU")):
    """Reads SKU data from the CSV file and returns a list of dictionaries.

    If you are asked for the available items or any enquiry about items/SKUs,
    call this function and return only consumer-friendly information like
    id, name, cost, and available quantity.
    """
    if not os.path.exists(CSV_FILE_PATH):
        return {"error": "SKU data file not found."}

    skus = []
    try:
        with open(CSV_FILE_PATH, newline="", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                if (
                    sku_name == "*"
                    or sku_name.lower()
                    in row.get("ProductName", "").lower()
                ):
                    skus.append(row)
        return {"skus": skus}
    except (OSError, csv.Error) as e:
        return {"error": f"Failed to read SKU data: {str(e)}"}


#### Inventory Mutation Tool (`update_sku_qty`) and Server Entrypoint

Finally, we define our second MCP tool, `update_sku_qty`, which allows an agent to modify inventory quantities and automatically update stock status. We also add the server entrypoint.

* **State Mutation via Tool**: The tool accepts `sku_id`, `quantity`, and `sign` (`1` for adding, `-1` for removing). It reads `sku_data.csv`, updates `QuantityOnHand` for the matching SKU, and automatically recalculates whether `Status` should be `'Low Stock'` or `'In Stock'` based on `ReorderLevel`.
* **Schema & Validation**: Pydantic `Field(..., ge=0)` ensures that `quantity` must be a non-negative integer.
* **Server Entrypoint**: Under `if __name__ == "__main__":`, we invoke `mcp.run(transport="sse")` to start the server using the **Server-Sent Events (SSE)** transport protocol.

In [ ]:
%%writefile -a ./custom_mcp_agent/mcp_server.py


@mcp.tool()
def update_sku_qty(
    sku_id: str = Field(
        ..., description="The SKU ID of the product to update."
    ),
    quantity: int = Field(
        ...,
        description="The quantity to be added or removed from the SKU.",
        ge=0,
    ),
    sign: int = Field(1, description="1 for adding and -1 for removing."),
):
    """Updates the quantity of a specific SKU in the CSV file.

    If you are asked for placing or returning/cancelling an order, call this
    function.

    Args:
        sku_id (str): The SKU ID of the product to update.
        quantity (int): The new quantity for the SKU.
        sign (int): 1 for adding and -1 for removing.

    Returns:
        dict: A message indicating success or failure.
    """
    if not os.path.exists(CSV_FILE_PATH):
        return {"error": "SKU data file not found."}

    if not isinstance(quantity, int):
        return {"error": "Invalid quantity. Must be a non-negative integer."}

    rows = []
    updated = False
    updated_qty = 0
    fieldnames = []

    try:
        with open(CSV_FILE_PATH, newline="", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile)
            fieldnames = reader.fieldnames
            if not fieldnames:  # Handle empty or malformed CSV
                return {"error": "CSV file is empty or has no header."}

            quantity *= sign

            for row in reader:
                if row.get("SKU") == sku_id:
                    row["QuantityOnHand"] = int(row["QuantityOnHand"]) + int(
                        quantity
                    )
                    # Potentially update "Status" based on
                    # new quantity vs ReorderLevel
                    if "ReorderLevel" in row and quantity <= int(
                        row.get("ReorderLevel", 0)
                    ):
                        row["Status"] = "Low Stock"
                    elif "ReorderLevel" in row and quantity > int(
                        row.get("ReorderLevel", 0)
                    ):
                        row["Status"] = "In Stock"
                    updated = True
                    updated_qty = row["QuantityOnHand"]
                rows.append(row)

        if not updated:
            return {"error": f"SKU ID '{sku_id}' not found."}

        with open(
            CSV_FILE_PATH, mode="w", newline="", encoding="utf-8"
        ) as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        return {
            "message": f"Quantity for SKU '{sku_id}' updated to {updated_qty}."
        }
    except (OSError, csv.Error, ValueError, KeyError) as e:
        return {"error": f"Failed to update SKU quantity: {str(e)}"}


if __name__ == "__main__":
    mcp.run(transport="sse")


### Start the Inventory MCP Server Locally

We now launch `mcp_server.py` as a background subprocess so that our local ADK agent can connect to its `/sse` endpoint on port `4200`. We'll verify that the server binds successfully before proceeding.


In [ ]:
# Launch the MCP server as a background subprocess
print("Starting Inventory MCP Server on port 4200...")
server_process = subprocess.Popen(
    [sys.executable, "-m", "custom_mcp_agent.mcp_server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait up to 15 seconds for the server to bind to port 4200
for _ in range(30):
    try:
        with socket.create_connection(("0.0.0.0", 4200), timeout=1):
            print("SUCCESS: MCP Server is running and listening on http://0.0.0.0:4200/sse")
            break
    except OSError:
        time.sleep(0.5)
else:
    print("ERROR: Failed to start MCP Server on port 4200.")


### Define the ADK Agent with `McpToolset` (`agent.py`)

In ADK, an agent connects to an MCP server by adding an `McpToolset` to its `tools` list.
* `McpToolset` handles connecting to the MCP server, discovering its available tools, and translating between Gemini function-calling declarations and MCP tool invocations.
* Here, we configure `McpToolset` with `SseServerParams(url="http://0.0.0.0:4200/sse")` so it connects to our local inventory MCP server over Server-Sent Events (SSE).


In [ ]:
%%writefile ./custom_mcp_agent/agent/agent.py
"""Example of using MCP Toolset to create an inventory management tool."""

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

MODEL = "gemini-3.5-flash"

# TODO: IMPORTANT! Change the path below to your remote MCP Server path
MCP_SERVER_URL = "http://0.0.0.0:4200/sse"

root_agent = LlmAgent(
    model=Gemini(
        model=MODEL,
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    name="inventory_assistant",
    description="You are a specialized assistant for inventory management.",
    instruction=(
        "Help user get answer to their queries about inventory and update "
        "or process the items."
    ),
    tools=[
        McpToolset(
            connection_params=SseServerParams(url=MCP_SERVER_URL),
        )
    ],
)


### Open the ADK Developer UI

You can launch the ADK Developer UI to interact with your MCP-powered agent using the `adk web` command.

Execute the cell below and open the printed URL. (It may take a few seconds for the web server to start.) Once the web UI opens, select **`agent`** from the agent dropdown at the top and try sending the following sample questions:

* *"Can you list all available SKUs along with their product names, cost, and quantities on hand?"* (Tests `list_skus`)
* *"What is the current stock level and status of SKU001 (Laptop Pro 15)?"* (Tests `list_skus`)
* *"We just received a new shipment of 15 units for SKU001. Please add 15 to the quantity of SKU001."* (Tests `update_sku_qty`)
* *"Which products are currently marked as Low Stock?"* (Tests `list_skus` filtering)

**Note**: You can also run `adk web` via the terminal. If you do so, ensure your virtual environment is activated first:
```
# in the asl-ml-immersion directory
source ./asl_genai/.venv/bin/activate
adk web ./asl_genai/notebooks/building_agents/solutions/custom_mcp_agent
```


In [ ]:
# On Cloud Workstations
!adk web custom_mcp_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**Note:** If you are using Agent Platform Workbench, remove the comment out and run the cell below.


In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web custom_mcp_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


### Verify Data Persistence in CSV File

After testing the agent in the Developer UI and interrupting the kernel, run the cell below to verify that the quantity for `SKU001` was updated in the underlying file.


In [ ]:
df = pd.read_csv("./custom_mcp_agent/sku_data.csv")
df


### Clean Up Background MCP Server

The local MCP server should have been terminated already when you interrupted the Jupyter kernel after running `adk web`, but let's run the cell below to make sure any remaining background `mcp_server` process listening on port `4200` is cleanly terminated before we proceed to deployment.


In [ ]:
if 'server_process' in locals() and server_process.poll() is None:
    print("Terminating Inventory MCP Server...")
    server_process.terminate()
    server_process.wait()
    print("SUCCESS: Inventory MCP Server terminated.")
else:
    print("MCP Server was not running or already terminated.")


## Deploy the MCP Server to Google Cloud Run

In an enterprise or production environment, you do not run MCP servers as local background subprocesses. Instead, you deploy your MCP server as a standalone microservice on **Google Cloud Run** so that any authorized AI agent across your organization can connect to its `/sse` endpoint securely over HTTPS.

In this section, we will:
1. Create container configuration files (`requirements.txt` and `Dockerfile`).
2. Deploy the Inventory MCP Server to Google Cloud Run using `gcloud run deploy`.
3. Retrieve the deployed Cloud Run service URL and connect a remote ADK agent to it step-by-step.


### Create Cloud Run Configuration Files

First, we create a `requirements.txt` file listing the SDK dependencies required by our MCP server (`mcp>=1.25.0,<2.0.0`, `pydantic>=2.0.0`, and `uvicorn>=0.30.0`), and a `Dockerfile` that packages `mcp_server.py` and `sku_data.csv` into a container image. Note that we constrain `mcp` to `<2.0.0` to ensure compatibility with `FastMCP`.

Notice that Cloud Run automatically injects the `PORT` environment variable (default `8080`), which our `FastMCP` server reads via `int(os.environ.get("PORT", 4200))`.


In [ ]:
%%writefile ./custom_mcp_agent/requirements.txt
mcp>=1.25.0,<2.0.0
pydantic>=2.0.0
uvicorn>=0.30.0


In [ ]:
%%writefile ./custom_mcp_agent/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app

ENV PORT=8080
EXPOSE 8080

CMD ["python", "mcp_server.py"]


### Deploy Service Using Cloud Run CLI

We use the Google Cloud SDK (`gcloud run deploy`) with the `--source` flag. Cloud Run automatically builds the container image from our `Dockerfile` and deploys the service to your Google Cloud project.

**Note:** Deployment to Cloud Run typically takes around 2–3 minutes. Once completed, `gcloud` prints the secure HTTPS service URL (e.g., `https://inventory-mcp-server-xxxxxx-uc.a.run.app`).


In [ ]:
%%bash
gcloud run deploy inventory-mcp-server \
    --source=./custom_mcp_agent \
    --region=$CLOUD_RUN_LOCATION \
    --project=$GOOGLE_CLOUD_PROJECT \
    --allow-unauthenticated \
    --quiet


### Retrieve Deployed Cloud Run Service URL

Once deployment completes, we use the `gcloud run services describe` CLI command to retrieve the live HTTPS URL of our deployed Cloud Run service and append `/sse` to form the full Model Context Protocol SSE endpoint URL. We store this in the `REMOTE_MCP_SERVER_URL` environment variable.


In [ ]:
# Retrieve the deployed Cloud Run service URL
import os

SERVICE_URL = !gcloud run services describe "inventory-mcp-server" \
  --platform managed \
  --region $CLOUD_RUN_LOCATION \
  --project $PROJECT_ID \
  --format "value(status.url)"

remote_sse_url = f"{SERVICE_URL[0]}/sse"
os.environ["REMOTE_MCP_SERVER_URL"] = remote_sse_url
print("SUCCESS: Remote MCP Server SSE URL:", remote_sse_url)

# Persist REMOTE_MCP_SERVER_URL to custom_mcp_agent/.env file
env_file = "./custom_mcp_agent/.env"
if os.path.exists(env_file):
    with open(env_file, "a") as f:
        f.write(f"\nREMOTE_MCP_SERVER_URL={remote_sse_url}\n")


### Update ADK Agent to Connect to Cloud Run (`agent.py`)

Now that our MCP server is deployed to Google Cloud Run, let's update `./custom_mcp_agent/agent/agent.py` so that our agent connects to the remote Cloud Run `/sse` endpoint over HTTPS instead of localhost (`http://0.0.0.0:4200/sse`).

Notice how `MCP_SERVER_URL` reads `REMOTE_MCP_SERVER_URL` from the environment, falling back to the Cloud Run service URL.

In [ ]:
%%writefile ./custom_mcp_agent/agent/agent.py
"""Example of using MCP Toolset to create an inventory management tool."""

import os

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

# Load environment variables from .env file
env_path = os.path.abspath(
    os.path.join(os.path.dirname(__file__), "..", ".env")
)
if os.path.exists(env_path):
    load_dotenv(env_path, override=True)
else:
    load_dotenv(override=True)

MODEL = "gemini-3.5-flash"

# Connect to the remote Cloud Run MCP Server over HTTPS
MCP_SERVER_URL = os.environ.get(
    "REMOTE_MCP_SERVER_URL",
    "https://inventory-mcp-server-xxxxxxxx-uc.a.run.app/sse",
)

root_agent = LlmAgent(
    model=Gemini(
        model=MODEL,
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    name="inventory_assistant",
    description="You are a specialized assistant for inventory management.",
    instruction=(
        "Help user get answer to their queries about inventory and update "
        "or process the items."
    ),
    tools=[
        McpToolset(
            connection_params=SseServerParams(url=MCP_SERVER_URL),
        )
    ],
)

### Test the Remote Cloud Run MCP Agent

Let's test our updated `inventory_assistant` agent programmatically using ADK's `Runner` to verify that our agent successfully connects to Google Cloud Run over HTTPS and calls the remote `list_skus` tool!


In [ ]:
from custom_mcp_agent.agent import agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
import importlib

importlib.reload(agent)

USER_ID = "user_1"
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name="remote_mcp_test", user_id=USER_ID, session_id="remote_s1"
)

remote_runner = Runner(
    agent=agent.root_agent,
    app_name="remote_mcp_test",
    session_service=session_service,
)

query = "Can you list all available SKUs along with their product names, cost, and quantities on hand?"
print(f"\n=======================================================")
print(f">>> User Query: {query}")
print(f"=======================================================")

content = types.Content(role="user", parts=[types.Part(text=query)])
final_response_text = "Agent did not produce a final response."

async for event in remote_runner.run_async(
    user_id=USER_ID, session_id="remote_s1", new_message=content
):
    if event.is_final_response():
        if event.content and event.content.parts:
            final_response_text = event.content.parts[0].text
        break

print(f"<<< Agent Response:\n{final_response_text}\n")


### Inspect the Remote Agent in ADK Developer UI

You can also test and inspect your remote Cloud Run MCP agent using the graphical Developer UI (`adk web`):

1. Execute the cell below to launch the ADK Developer UI.
2. Open the printed URL and select **`agent`** from the agent dropdown (now powered by your remote Cloud Run MCP server!).
3. Try sending sample questions to test your Cloud Run MCP service:
   * *"What is the current stock level and status of SKU001 (Laptop Pro 15)?"*
   * *"Can you list all available SKUs along with their quantities on hand?"*
   * *"We just received 5 more units of SKU001. Please add 5 to the quantity of SKU001."*
4. Inspect the remote HTTPS SSE tool invocations in the **Events** panel.
5. **Check Cloud Run Metrics**: After interacting with the remote agent, check the [Cloud Run Observability Page](https://console.cloud.google.com/run/detail/us-central1/inventory-mcp-server/observability/metrics) to observe the incoming request counts and container metrics generated by your agent's tool invocations.
6. Click **`Interrupt the kernel`** in Jupyter when you are finished exploring.


In [ ]:
# On Cloud Workstations
!adk web custom_mcp_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**Note:** If you are using Agent Platform Workbench, remove the comment out and run the cell below.


In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web custom_mcp_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


#### Observe Cloud Run Metrics in Google Cloud Console

After interacting with your remote MCP agent in the Developer UI, you can verify that your agent's tool calls were invoked on your remote server by checking Google Cloud Console:

* Open the **[Cloud Run Observability & Metrics Page](https://console.cloud.google.com/run/detail/us-central1/inventory-mcp-server/observability/metrics)** for `inventory-mcp-server`.
* Notice the spikes in **Request count**, **Container CPU utilization**, and **Request latency** corresponding to your SSE tool invocations from the Developer UI!


---

### Production Consideration: Securing Cloud Run with Service Accounts & IAM

For convenience, we deployed our Cloud Run service with `--allow-unauthenticated`. In production, you should **never** expose internal MCP servers publicly.

To secure your Cloud Run MCP service in production:
1. **Omit `--allow-unauthenticated`** 
2. **Grant Cloud Run Invoker (`roles/run.invoker`)** to your agent's Service Account:
   ```bash
   gcloud run services add-iam-policy-binding inventory-mcp-server \
       --region=$CLOUD_RUN_LOCATION \
       --project=$GOOGLE_CLOUD_PROJECT \
       --member="serviceAccount:YOUR_AGENT_SA@$GOOGLE_CLOUD_PROJECT.iam.gserviceaccount.com" \
       --role="roles/run.invoker"
   ```
3. **Pass an OIDC ID Token Header in `agent.py`**:
   Google ADK's `SseServerParams` natively accepts HTTP headers. You can fetch a signed OpenID Connect (OIDC) ID token targeting your Cloud Run audience and pass an `Authorization: Bearer <ID_TOKEN>` header:
   ```python
   from google.auth.transport.requests import Request
   from google.oauth2 import id_token
   from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
   from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

   # 1. Base audience for ID token (root Cloud Run URL without /sse path)
   audience = MCP_SERVER_URL.replace("/sse", "").rstrip("/")

   # 2. Fetch OIDC ID Token for Service Account authentication
   headers = {}
   try:
       auth_req = Request()
       token = id_token.fetch_id_token(auth_req, audience=audience)
       headers["Authorization"] = f"Bearer {token}"
   except Exception:
       pass

   # 3. Pass auth headers into SseServerParams
   mcp_toolset = McpToolset(
       connection_params=SseServerParams(
           url=MCP_SERVER_URL,
           headers=headers,
       )
   )
   ```


## Summary

Congratulations! You have successfully built, tested, and inspected a Generative AI agent using the **Model Context Protocol (MCP)** and Google's **Agent Development Kit (ADK)**:

* **MCP Server Integration**: Built an inventory management MCP server using `FastMCP` (`@mcp.tool()`) with both read-only (`list_skus`) and state-mutating (`update_sku_qty`) tools.
* **ADK McpToolset**: Connected an ADK `LlmAgent` to the local server over SSE using `McpToolset` and `SseServerParams`.
* **ADK Developer UI**: Launched `adk web` to interactively inspect and test agent-to-MCP communication and verify real-time inventory updates.
* **Cloud Run Production Deployment**: Packaged the MCP server with a `Dockerfile`, deployed it as a standalone cloud microservice using `gcloud run deploy`, and updated our ADK agent (`agent.py`) to connect to its remote HTTPS `/sse` endpoint.


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.